<a href="https://colab.research.google.com/github/gitmystuff/DTSC4050/blob/main/final_project_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final Project Template — Data Science Pipeline

**Use this notebook as a skeleton for your final project.** Replace the
example dataset and commentary with your own, but keep the section
structure — it maps directly onto a ~5 minute presentation (see the
timing guide at the bottom).

This template demonstrates the full workflow on two built-in scikit-learn
datasets so every cell runs out of the box, with **no internet download
required** (both datasets ship with scikit-learn):

- **Regression example:** Diabetes dataset (predict disease progression score)
- **Classification example:** Breast Cancer Wisconsin (predict malignant vs. benign)

Delete whichever branch (regression / classification) doesn't apply to
your project, or keep both if you want a reference for each.

**Sections**
1. Getting the Data
2. Exploratory Data Analysis (5 plots)
3. Data Prep & Cleaning
4. Feature Engineering
5. Feature Selection
6. Modeling (Linear Regression / Logistic Regression)
7. Model Evaluation
8. 5-Minute Presentation Guide


In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42


## 1. Getting the Data

Load your raw data here. A few common patterns are shown below —
uncomment the one that matches your source, or use the built-in
sklearn datasets used throughout this template.


In [ ]:
# --- Option A: built-in sklearn dataset (used in this template) ---
from sklearn.datasets import load_diabetes, load_breast_cancer

diabetes = load_diabetes(as_frame=True)
df_reg = diabetes.frame.copy()                 # regression dataset
TARGET_REG = "target"

cancer = load_breast_cancer(as_frame=True)
df_clf = cancer.frame.copy()                   # classification dataset
TARGET_CLF = "target"

# --- Option B: load your own file (uncomment & edit) ---
# df = pd.read_csv("data/my_data.csv")
# df = pd.read_excel("data/my_data.xlsx")
# df = pd.read_json("data/my_data.json")

# --- Option C: pull from a database / API (uncomment & edit) ---
# import sqlite3
# conn = sqlite3.connect("data/my_database.db")
# df = pd.read_sql("SELECT * FROM my_table", conn)

print("Regression dataset:", df_reg.shape)
print("Classification dataset:", df_clf.shape)
df_reg.head()


In [ ]:
df_clf.head()


**Train/test split first.** Splitting before EDA/cleaning avoids
leaking information from the test set into decisions made about the
training data (a classic source of overly-optimistic results).


In [ ]:
# Regression split
X_reg = df_reg.drop(columns=[TARGET_REG])
y_reg = df_reg[TARGET_REG]
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

# Classification split (stratify on the target to preserve class balance)
X_clf = df_clf.drop(columns=[TARGET_CLF])
y_clf = df_clf[TARGET_CLF]
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=RANDOM_STATE, stratify=y_clf
)

print("Regression train/test:", X_train_reg.shape, X_test_reg.shape)
print("Classification train/test:", X_train_clf.shape, X_test_clf.shape)


## 2. Exploratory Data Analysis (EDA)

Goal: understand distributions, relationships, and data quality issues
**before** you start modeling. For a presentation, pick 2-3 of the most
interesting/persuasive plots rather than showing all of them — but keep
all of them here in the notebook as your working analysis.

We'll use the **training data only** (`X_train`, `y_train`) throughout
EDA to avoid peeking at the test set.


In [ ]:
# Quick structural overview
X_train_reg.info()
print()
X_train_reg.describe().T


### Plot 1 — Distributions of numeric features

Histograms are the fastest way to spot skew, outliers, and weird
placeholder values (e.g. -999, 0 used as "missing").


In [ ]:
X_train_reg.hist(figsize=(12, 8), bins=30)
plt.suptitle("Distributions of Numeric Features (Training Set)", y=1.02)
plt.tight_layout()
plt.show()


### Plot 2 — Correlation heatmap

Shows pairwise linear relationships between all numeric features
(and the target, if you include it in the frame). Look for:
- Strong correlations with the target (candidate predictive features)
- Strong correlations between features (multicollinearity — relevant later
  for feature selection)


In [ ]:
train_reg_with_target = X_train_reg.copy()
train_reg_with_target[TARGET_REG] = y_train_reg

plt.figure(figsize=(10, 8))
corr = train_reg_with_target.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation Heatmap (Training Set)")
plt.tight_layout()
plt.show()


### Plot 3 — Feature correlation with the target

A bar chart of each feature's correlation with the target is often more
presentation-friendly than the full heatmap: it directly answers
"which features matter?"


In [ ]:
plt.figure(figsize=(8, 5))
X_train_reg.corrwith(y_train_reg).sort_values().plot.bar()
plt.title(f"Feature Correlation with {TARGET_REG}")
plt.ylabel("Pearson correlation")
plt.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()


### Plot 4 — Relationship between the strongest predictor and the target

A scatter plot (with a light regression line) makes the strongest
relationship from Plot 3 concrete.


In [ ]:
top_feature = X_train_reg.corrwith(y_train_reg).abs().idxmax()

plt.figure(figsize=(7, 5))
sns.regplot(x=X_train_reg[top_feature], y=y_train_reg,
            scatter_kws={"alpha": 0.2, "s": 10}, line_kws={"color": "red"})
plt.title(f"{top_feature} vs. {TARGET_REG} (strongest correlation)")
plt.tight_layout()
plt.show()


### Plot 5 — Outlier / spread check via boxplots

Boxplots (on standardized features, so they're comparable side by side)
highlight outliers and skew that histograms can undersell.


In [ ]:
from sklearn.preprocessing import StandardScaler

scaled_preview = pd.DataFrame(
    StandardScaler().fit_transform(X_train_reg),
    columns=X_train_reg.columns
)

plt.figure(figsize=(12, 5))
sns.boxplot(data=scaled_preview)
plt.xticks(rotation=45, ha="right")
plt.title("Standardized Feature Spread & Outliers (Training Set)")
plt.tight_layout()
plt.show()


**For a classification project**, two EDA additions that are usually
worth including: a class-balance count plot, and a correlation-with-target
bar chart using the numeric-encoded label (same `corrwith` pattern as above).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.countplot(x=y_train_clf, ax=axes[0])
axes[0].set_title("Class Balance (0 = malignant, 1 = benign)")
axes[0].set_xlabel(TARGET_CLF)

X_train_clf.corrwith(y_train_clf).sort_values().plot.bar(ax=axes[1])
axes[1].set_title(f"Feature Correlation with {TARGET_CLF}")
axes[1].axhline(0, color="black", linewidth=0.8)

plt.tight_layout()
plt.show()


## 3. Data Prep & Cleaning

Common cleaning steps — apply whichever your dataset actually needs.
**Fit any imputers/encoders/scalers on the training data only**, then
apply (`.transform`, not `.fit_transform`) to the test data, to avoid
data leakage.


In [ ]:
# --- Missing values ---
print("Missing values per column:\n", X_train_reg.isna().sum())

# Example imputation pattern (uncomment & adapt):
# from sklearn.impute import SimpleImputer
# num_imputer = SimpleImputer(strategy="median")
# X_train_reg[num_cols] = num_imputer.fit_transform(X_train_reg[num_cols])
# X_test_reg[num_cols]  = num_imputer.transform(X_test_reg[num_cols])

# --- Duplicates ---
print("Duplicate rows:", X_train_reg.duplicated().sum())
# X_train_reg = X_train_reg.drop_duplicates()

# --- Outlier handling (example: cap at 1st/99th percentile) ---
def cap_outliers(series, lower_q=0.01, upper_q=0.99):
    lo, hi = series.quantile([lower_q, upper_q])
    return series.clip(lo, hi)

X_train_reg_clean = X_train_reg.copy()
for col in X_train_reg_clean.columns:
    X_train_reg_clean[col] = cap_outliers(X_train_reg_clean[col])

# --- Categorical encoding (if you have categorical columns) ---
# X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
# X_test  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=True)
# X_test  = X_test.reindex(columns=X_train.columns, fill_value=0)  # align columns

X_train_reg_clean.describe().T


## 4. Feature Engineering

Create new features that might carry more signal than the raw columns,
and scale numeric features (important for linear/logistic regression,
which are sensitive to feature magnitude).


In [ ]:
# --- Example derived feature (Diabetes dataset) ---
X_train_fe = X_train_reg_clean.copy()
X_test_fe = X_test_reg.copy()

# Interaction term: BMI x blood pressure, a plausible combined risk signal
X_train_fe["bmi_bp_interaction"] = X_train_fe["bmi"] * X_train_fe["bp"]
X_test_fe["bmi_bp_interaction"] = X_test_fe["bmi"] * X_test_fe["bp"]

# --- Scaling (fit on train, apply to test) ---
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_fe), columns=X_train_fe.columns, index=X_train_fe.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_fe), columns=X_test_fe.columns, index=X_test_fe.index
)

X_train_scaled.head()


## 5. Feature Selection

A few common approaches — pick what fits your project and dataset size:

1. **Correlation filter** — drop features weakly correlated with the target
   or highly correlated with each other (redundant).
2. **Statistical test (`SelectKBest`)** — keep the k features most
   related to the target by a statistical score.
3. **Model-based importance** — fit a quick model and rank features by
   coefficient magnitude / importance.


In [ ]:
from sklearn.feature_selection import SelectKBest, f_regression

# Statistical selection: top-k features by F-statistic vs. the target
k = 5
selector = SelectKBest(score_func=f_regression, k=k)
selector.fit(X_train_scaled, y_train_reg)

selected_features = X_train_scaled.columns[selector.get_support()]
print(f"Top {k} selected features:", list(selected_features))

X_train_selected = X_train_scaled[selected_features]
X_test_selected = X_test_scaled[selected_features]


## 6. Modeling

### 6a. Linear Regression (for a numeric target)


In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train_selected, y_train_reg)

y_pred_reg = lin_reg.predict(X_test_selected)

coef_table = pd.Series(lin_reg.coef_, index=X_train_selected.columns).sort_values()
print("Intercept:", lin_reg.intercept_)
coef_table


### 6b. Logistic Regression (for a binary/categorical target)

Reusing the classification dataset (`X_train_clf` / `y_train_clf`) loaded
in Section 1. The same scale → select → fit pattern applies.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

# Scale
clf_scaler = StandardScaler()
X_train_clf_scaled = pd.DataFrame(
    clf_scaler.fit_transform(X_train_clf), columns=X_train_clf.columns, index=X_train_clf.index
)
X_test_clf_scaled = pd.DataFrame(
    clf_scaler.transform(X_test_clf), columns=X_test_clf.columns, index=X_test_clf.index
)

# Select top features
clf_selector = SelectKBest(score_func=f_classif, k=8)
clf_selector.fit(X_train_clf_scaled, y_train_clf)
clf_features = X_train_clf_scaled.columns[clf_selector.get_support()]

X_train_clf_selected = X_train_clf_scaled[clf_features]
X_test_clf_selected = X_test_clf_scaled[clf_features]

# Fit
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_clf_selected, y_train_clf)

y_pred_clf = log_reg.predict(X_test_clf_selected)
y_proba_clf = log_reg.predict_proba(X_test_clf_selected)[:, 1]

print("Selected features:", list(clf_features))


## 7. Model Evaluation

Use the metrics that match your model type:

- **Regression:** RMSE, MAE, R²
- **Classification:** Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix


In [ ]:
# --- Regression metrics ---
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rmse = mean_squared_error(y_test_reg, y_pred_reg) ** 0.5
mae = mean_absolute_error(y_test_reg, y_pred_reg)
r2 = r2_score(y_test_reg, y_pred_reg)

print(f"RMSE: {rmse:.3f}")
print(f"MAE:  {mae:.3f}")
print(f"R^2:  {r2:.3f}")

# Residual plot — a quick visual check for model bias / heteroscedasticity
residuals = y_test_reg - y_pred_reg
plt.figure(figsize=(7, 5))
plt.scatter(y_pred_reg, residuals, alpha=0.3, s=10)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted value")
plt.ylabel("Residual (actual - predicted)")
plt.title("Residual Plot — Linear Regression")
plt.tight_layout()
plt.show()


In [ ]:
# --- Classification metrics ---
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, RocCurveDisplay
)

print(f"Accuracy:  {accuracy_score(y_test_clf, y_pred_clf):.3f}")
print(f"Precision: {precision_score(y_test_clf, y_pred_clf):.3f}")
print(f"Recall:    {recall_score(y_test_clf, y_pred_clf):.3f}")
print(f"F1 score:  {f1_score(y_test_clf, y_pred_clf):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test_clf, y_proba_clf):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test_clf, y_pred_clf)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=cancer.target_names, yticklabels=cancer.target_names)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Confusion Matrix")

RocCurveDisplay.from_predictions(y_test_clf, y_proba_clf, ax=axes[1])
axes[1].set_title("ROC Curve")

plt.tight_layout()
plt.show()


## 8. Five-Minute Presentation Guide

A tight time budget for a ~5 minute talk built from this notebook:

| Time | Section | What to say |
|---|---|---|
| 0:00–0:30 | **Problem & data** | What are you predicting, and why does it matter? Where did the data come from, how many rows/features? |
| 0:30–1:30 | **Key EDA insight** | Show 1-2 plots (e.g. the `corrwith` bar chart + correlation heatmap or the strongest scatter plot). Say what pattern you found, not everything you tried. |
| 1:30–2:15 | **Cleaning & feature engineering** | One sentence on data quality issues you fixed, one on the most useful engineered feature. |
| 2:15–2:45 | **Feature selection** | Which features you kept and why (statistical test / correlation / importance) — this shows deliberate reasoning, not just "used everything." |
| 2:45–3:45 | **Model** | Which model, why it's an appropriate choice for the problem (regression vs. classification), and how you evaluated it (train/test split). |
| 3:45–4:30 | **Results** | Lead with 1-2 headline metrics (e.g. R² / RMSE, or accuracy + ROC-AUC) and one supporting plot (residuals, or confusion matrix / ROC curve). |
| 4:30–5:00 | **Takeaway & limitations** | One sentence: what would you do with more time (more features, a different model, more data)? |

**Tips**
- Don't walk through every cell — pick the plots/metrics that tell the story.
- Practice the timing out loud once; 5 minutes goes fast.
- Have 1-2 backup slides/cells ready for likely questions (e.g. "why not use feature X?", "what about outliers?").
